In [1]:
import os
import torch
import pandas as pd
import tarfile
import shutil

from tqdm.auto import tqdm

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Run Model on: {DEVICE}")

Run Model on: cuda


In [3]:
chunk_number = 91
tarfile_path = fr"dataset\urban-frdr\audio_2022_chunk_{chunk_number}.tar.gz"
csv_frdr = r"dataset\urban-frdr\inspections_2022.csv"
csv_zenodo = r"dataset\zenodo\state_labels.csv"
frdr_audiofolder_path = fr"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"
zenodo_audiofolder_path = r"dataset\zenodo"
output_folder = f"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"

In [ ]:
def process_frdr_audio(tarfile_path, csv_frdr, frdr_audiofolder_path, output_folder):
    with tarfile.open(tarfile_path, "r:gz") as tar:
        tar.extractall(path=output_folder)

    df_frdr = pd.read_csv(csv_frdr)
    target = df_frdr[(df_frdr['Queen status'] == 'queenless') | (df_frdr['Category'] == 'varroa')]
    print(target[['Date', 'Tag number', 'Category', 'Queen status']])

    all_files = os.listdir(frdr_audiofolder_path)
    extract_list = []
    for file_path in all_files:
        if file_path.endswith('.wav'):
            file_name = os.path.basename(file_path)
            parts = file_name.split('_')
            if len(parts) < 3:
                continue
            date = parts[0]
            hive_part = parts[2]
            hive_id = hive_part.split('-')[1].split('.')[0]
            for_merge = pd.to_datetime(date, format='%d-%m-%Y').strftime('%Y-%m-%d')
            extract_list.append({
                'Filename': file_name,
                'Date': for_merge,
                'Tag number': int(hive_id)
            })

    df_audio = pd.DataFrame(extract_list)
    df_audio['Tag number'] = df_audio['Tag number'].astype(int)

    df_frdr['Tag number'] = df_frdr['Tag number'].astype(int)
    df_frdr['Date'] = pd.to_datetime(df_frdr['Date']).dt.strftime('%Y-%m-%d')

    df_frdr_sorted = df_frdr.sort_values(by=['Date', 'Tag number', 'Category'], ascending=[True, True, False])
    df_frdr_clean = df_frdr_sorted.drop_duplicates(subset=['Date', 'Tag number'], keep='first')
    merge_df = pd.merge(df_audio, df_frdr_clean, on=['Date', 'Tag number'], how='inner')

    def check_class(row):
        if row['Category'] == 'varroa':
            return 'Class 2: Infested'
        elif row['Queen status'] == 'queenless':
            return 'Class 1: Queenless'
        elif row['Queen status'] == 'queenright' and row['Is alive'] == 1:
            return 'Class 0: Active'
        else:
            return 'Others / Ignored'

    merge_df['target_Class'] = merge_df.apply(check_class, axis=1)

    # for index, row in merge_df.iterrows():
    #     print(f"{index}. File: {row['Filename']} | {row['target_Class']}")
    # print(merge_df['target_Class'].value_counts())

    unmapped_folder_path = os.path.join(os.path.dirname(frdr_audiofolder_path), "unmapped_audio")
    os.makedirs(unmapped_folder_path, exist_ok=True)
    frdr_class_map = {}
    for index, row in merge_df.iterrows():
        old_name = row['Filename']
        current_class = row['target_Class']
        if current_class == 'Others / Ignored':
            file_to_delete = os.path.join(frdr_audiofolder_path, old_name)
            if os.path.exists(file_to_delete):
                os.remove(file_to_delete)
            continue
        class_suffix = current_class.split(': ')[1]
        frdr_class_map[old_name] = class_suffix

    rename_count = 0
    move_count = 0
    for filename in tqdm(os.listdir(frdr_audiofolder_path), total=len(os.listdir(frdr_audiofolder_path))):
        if filename.endswith('.wav'):
            old_file_path = os.path.join(frdr_audiofolder_path, filename)
            if filename in frdr_class_map:
                class_suffix = frdr_class_map[filename]
                name_part, ext_part = os.path.splitext(filename)
                new_filename = f"{name_part}_{class_suffix}{ext_part}"
                new_file_path = os.path.join(frdr_audiofolder_path, new_filename)
                if os.path.exists(old_file_path):
                    os.rename(old_file_path, new_file_path)
                    rename_count += 1
            else:
                destination_file_path = os.path.join(unmapped_folder_path, filename)
                if os.path.exists(old_file_path):
                    shutil.move(old_file_path, destination_file_path)
                    move_count += 1

    # print(f"Found: {rename_count} Files")
    return merge_df

merge_df = process_frdr_audio(tarfile_path, csv_frdr, frdr_audiofolder_path, output_folder)

C:\Users\USER\AppData\Local\Temp\ipykernel_11680\451829813.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=output_folder)


                                 Date  Tag number        Category Queen status
17   2022-08-24 20:30:15.427454+00:00        3627          varroa   queenright
31   2022-07-11 20:23:05.641387+00:00        3692     hive status    queenless
32   2022-07-11 20:23:05.641387+00:00        3692    hive grading    queenless
33   2022-07-11 20:23:05.641387+00:00        3692  frames of bees    queenless
34   2022-07-19 21:32:34.257822+00:00        3692     hive status    queenless
35   2022-07-19 21:32:34.257822+00:00        3692    hive grading    queenless
44   2022-08-24 20:16:33.633384+00:00        3692          varroa   queenright
71   2022-08-24 20:53:05.475343+00:00        3691          varroa   queenright
77   2022-09-30 21:47:53.014515+00:00        3691          varroa   queenright
98   2022-08-24 21:11:39.506501+00:00        3628          varroa   queenright
105  2022-09-30 21:29:31.817886+00:00        3628          varroa   queenright
108  2022-06-02 17:13:07.060900+00:00        3693   

  0%|          | 0/381 [00:00<?, ?it/s]

In [5]:
chunk_number = 90
tarfile_path = fr"dataset\urban-frdr\audio_2022_chunk_{chunk_number}.tar.gz"
csv_frdr = r"dataset\urban-frdr\inspections_2022.csv"
csv_zenodo = r"dataset\zenodo\state_labels.csv"
frdr_audiofolder_path = fr"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"
zenodo_audiofolder_path = r"dataset\zenodo"
output_folder = f"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"

In [6]:
chunk_number = 92
tarfile_path = fr"dataset\urban-frdr\audio_2022_chunk_{chunk_number}.tar.gz"
csv_frdr = r"dataset\urban-frdr\inspections_2022.csv"
csv_zenodo = r"dataset\zenodo\state_labels.csv"
frdr_audiofolder_path = fr"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"
zenodo_audiofolder_path = r"dataset\zenodo"
output_folder = f"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"

In [7]:
process_frdr_audio(tarfile_path, csv_frdr, frdr_audiofolder_path, output_folder)

C:\Users\USER\AppData\Local\Temp\ipykernel_11680\451829813.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=output_folder)


                                 Date  Tag number        Category Queen status
17   2022-08-24 20:30:15.427454+00:00        3627          varroa   queenright
31   2022-07-11 20:23:05.641387+00:00        3692     hive status    queenless
32   2022-07-11 20:23:05.641387+00:00        3692    hive grading    queenless
33   2022-07-11 20:23:05.641387+00:00        3692  frames of bees    queenless
34   2022-07-19 21:32:34.257822+00:00        3692     hive status    queenless
35   2022-07-19 21:32:34.257822+00:00        3692    hive grading    queenless
44   2022-08-24 20:16:33.633384+00:00        3692          varroa   queenright
71   2022-08-24 20:53:05.475343+00:00        3691          varroa   queenright
77   2022-09-30 21:47:53.014515+00:00        3691          varroa   queenright
98   2022-08-24 21:11:39.506501+00:00        3628          varroa   queenright
105  2022-09-30 21:29:31.817886+00:00        3628          varroa   queenright
108  2022-06-02 17:13:07.060900+00:00        3693   

  0%|          | 0/381 [00:00<?, ?it/s]

,Filename,Date,Tag number,Category,Action detail,Queen status,Is alive,Report notes,target_Class
0,02-06-2022_06h15_HIVE-3631.wav,2022-06-02,3631,hive status,queenright,queenright,1,third box almost filled. added queeen excluder,Class 0: Active
1,02-06-2022_23h00_HIVE-6.wav,2022-06-02,6,hive status,queenless,queenless,1,no queen cells or eggs. need to add eggs frame...,Class 1: Queenless
2,04-07-2022_15h30_HIVE-3690.wav,2022-07-04,3690,hive status,queenright,queenright,1,18 fob,Class 0: Active
3,04-08-2022_13h15_HIVE-3690.wav,2022-08-04,3690,hive grading,pulled honey super,queenright,1,pulled 6 honey supers,Class 0: Active
4,04-08-2022_17h15_HIVE-3627.wav,2022-08-04,3627,hive grading,pulled honey super,queenright,1,pulled 6 honey supers,Class 0: Active
5,07-09-2022_21h30_HIVE-3631.wav,2022-09-07,3631,frames of bees,10,queenright,1,NaN,Class 0: Active
6,11-07-2022_15h30_HIVE-3693.wav,2022-07-11,3693,hive status,queenright,queenright,1,NaN,Class 0: Active
7,11-11-2022_16h15_HIVE-3631.wav,2022-11-11,3631,NaN,NaN,queenright,1,Added V3 sensor 217,Class 0: Active
8,11-11-2022_21h00_HIVE-3627.wav,2022-11-11,3627,NaN,NaN,queenright,1,Added V3 sensor 218,Class 0: Active
9,15-09-2022_14h45_HIVE-3628.wav,2022-09-15,3628,feeding,sugar,queenright,1,NaN,Class 0: Active


In [8]:
chunk_number = 93
tarfile_path = fr"dataset\urban-frdr\audio_2022_chunk_{chunk_number}.tar.gz"
csv_frdr = r"dataset\urban-frdr\inspections_2022.csv"
csv_zenodo = r"dataset\zenodo\state_labels.csv"
frdr_audiofolder_path = fr"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"
zenodo_audiofolder_path = r"dataset\zenodo"
output_folder = f"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"

In [9]:
process_frdr_audio(tarfile_path, csv_frdr, frdr_audiofolder_path, output_folder)

C:\Users\USER\AppData\Local\Temp\ipykernel_11680\451829813.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=output_folder)


                                 Date  Tag number        Category Queen status
17   2022-08-24 20:30:15.427454+00:00        3627          varroa   queenright
31   2022-07-11 20:23:05.641387+00:00        3692     hive status    queenless
32   2022-07-11 20:23:05.641387+00:00        3692    hive grading    queenless
33   2022-07-11 20:23:05.641387+00:00        3692  frames of bees    queenless
34   2022-07-19 21:32:34.257822+00:00        3692     hive status    queenless
35   2022-07-19 21:32:34.257822+00:00        3692    hive grading    queenless
44   2022-08-24 20:16:33.633384+00:00        3692          varroa   queenright
71   2022-08-24 20:53:05.475343+00:00        3691          varroa   queenright
77   2022-09-30 21:47:53.014515+00:00        3691          varroa   queenright
98   2022-08-24 21:11:39.506501+00:00        3628          varroa   queenright
105  2022-09-30 21:29:31.817886+00:00        3628          varroa   queenright
108  2022-06-02 17:13:07.060900+00:00        3693   

  0%|          | 0/381 [00:00<?, ?it/s]

,Filename,Date,Tag number,Category,Action detail,Queen status,Is alive,Report notes,target_Class
0,02-06-2022_09h45_HIVE-3631.wav,2022-06-02,3631,hive status,queenright,queenright,1,third box almost filled. added queeen excluder,Class 0: Active
1,04-07-2022_15h30_HIVE-3691.wav,2022-07-04,3691,hive grading,strong,queenright,1,25 fob. added second honey box (4th),Class 0: Active
2,04-08-2022_13h30_HIVE-3628.wav,2022-08-04,3628,hive grading,pulled honey super,queenright,1,pulled 6 honey supers,Class 0: Active
3,04-08-2022_17h15_HIVE-3690.wav,2022-08-04,3690,hive grading,pulled honey super,queenright,1,pulled 6 honey supers,Class 0: Active
4,07-06-2022_18h30_HIVE-3693.wav,2022-06-07,3693,hive status,queenless,queenless,1,saw queen cells,Class 1: Queenless
5,07-09-2022_21h45_HIVE-6.wav,2022-09-07,6,frames of bees,8,queenright,1,NaN,Class 0: Active
6,11-07-2022_15h30_HIVE-6.wav,2022-07-11,6,hive status,queenless,queenless,1,NaN,Class 1: Queenless
7,11-11-2022_05h15_HIVE-3631.wav,2022-11-11,3631,NaN,NaN,queenright,1,Added V3 sensor 217,Class 0: Active
8,11-11-2022_16h15_HIVE-3640.wav,2022-11-11,3640,NaN,NaN,queenright,1,Added V3 sensor 213,Class 0: Active
9,11-11-2022_21h00_HIVE-3640.wav,2022-11-11,3640,NaN,NaN,queenright,1,Added V3 sensor 213,Class 0: Active


In [10]:
chunk_number = 94
tarfile_path = fr"dataset\urban-frdr\audio_2022_chunk_{chunk_number}.tar.gz"
csv_frdr = r"dataset\urban-frdr\inspections_2022.csv"
csv_zenodo = r"dataset\zenodo\state_labels.csv"
frdr_audiofolder_path = fr"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"
zenodo_audiofolder_path = r"dataset\zenodo"
output_folder = f"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"

In [11]:
process_frdr_audio(tarfile_path, csv_frdr, frdr_audiofolder_path, output_folder)

C:\Users\USER\AppData\Local\Temp\ipykernel_11680\451829813.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=output_folder)


                                 Date  Tag number        Category Queen status
17   2022-08-24 20:30:15.427454+00:00        3627          varroa   queenright
31   2022-07-11 20:23:05.641387+00:00        3692     hive status    queenless
32   2022-07-11 20:23:05.641387+00:00        3692    hive grading    queenless
33   2022-07-11 20:23:05.641387+00:00        3692  frames of bees    queenless
34   2022-07-19 21:32:34.257822+00:00        3692     hive status    queenless
35   2022-07-19 21:32:34.257822+00:00        3692    hive grading    queenless
44   2022-08-24 20:16:33.633384+00:00        3692          varroa   queenright
71   2022-08-24 20:53:05.475343+00:00        3691          varroa   queenright
77   2022-09-30 21:47:53.014515+00:00        3691          varroa   queenright
98   2022-08-24 21:11:39.506501+00:00        3628          varroa   queenright
105  2022-09-30 21:29:31.817886+00:00        3628          varroa   queenright
108  2022-06-02 17:13:07.060900+00:00        3693   

  0%|          | 0/381 [00:00<?, ?it/s]

,Filename,Date,Tag number,Category,Action detail,Queen status,Is alive,Report notes,target_Class
0,02-06-2022_11h15_HIVE-3690.wav,2022-06-02,3690,hive status,queenless,queenless,1,added egg frame from 258604,Class 1: Queenless
1,04-07-2022_15h45_HIVE-3628.wav,2022-07-04,3628,hive status,queenright,queenright,1,installed Q excluder. 14 fob,Class 0: Active
2,04-08-2022_13h30_HIVE-3692.wav,2022-08-04,3692,hive grading,pulled honey super,queenright,1,pulled 6 honey supers,Class 0: Active
3,04-08-2022_17h15_HIVE-3691.wav,2022-08-04,3691,hive grading,pulled honey super,queenright,1,pulled 6 honey supers,Class 0: Active
4,07-09-2022_03h00_HIVE-3631.wav,2022-09-07,3631,frames of bees,10,queenright,1,NaN,Class 0: Active
5,07-09-2022_22h00_HIVE-3631.wav,2022-09-07,3631,frames of bees,10,queenright,1,NaN,Class 0: Active
6,11-11-2022_04h30_HIVE-3640.wav,2022-11-11,3640,NaN,NaN,queenright,1,Added V3 sensor 213,Class 0: Active
7,11-11-2022_21h00_HIVE-3690.wav,2022-11-11,3690,NaN,NaN,queenright,1,Added V3 sensor 214,Class 0: Active
8,20-06-2022_01h00_HIVE-3640.wav,2022-06-20,3640,hive grading,strong,queenright,1,varroa,Class 0: Active
9,27-06-2022_14h45_HIVE-3691.wav,2022-06-27,3691,hive status,queenright,queenright,1,3rd box full of honey,Class 0: Active


In [12]:
chunk_number = 95
tarfile_path = fr"dataset\urban-frdr\audio_2022_chunk_{chunk_number}.tar.gz"
csv_frdr = r"dataset\urban-frdr\inspections_2022.csv"
csv_zenodo = r"dataset\zenodo\state_labels.csv"
frdr_audiofolder_path = fr"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"
zenodo_audiofolder_path = r"dataset\zenodo"
output_folder = f"dataset/urban-frdr/audio_2022_chunk_{chunk_number}"
process_frdr_audio(tarfile_path, csv_frdr, frdr_audiofolder_path, output_folder)

C:\Users\USER\AppData\Local\Temp\ipykernel_11680\451829813.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=output_folder)


                                 Date  Tag number        Category Queen status
17   2022-08-24 20:30:15.427454+00:00        3627          varroa   queenright
31   2022-07-11 20:23:05.641387+00:00        3692     hive status    queenless
32   2022-07-11 20:23:05.641387+00:00        3692    hive grading    queenless
33   2022-07-11 20:23:05.641387+00:00        3692  frames of bees    queenless
34   2022-07-19 21:32:34.257822+00:00        3692     hive status    queenless
35   2022-07-19 21:32:34.257822+00:00        3692    hive grading    queenless
44   2022-08-24 20:16:33.633384+00:00        3692          varroa   queenright
71   2022-08-24 20:53:05.475343+00:00        3691          varroa   queenright
77   2022-09-30 21:47:53.014515+00:00        3691          varroa   queenright
98   2022-08-24 21:11:39.506501+00:00        3628          varroa   queenright
105  2022-09-30 21:29:31.817886+00:00        3628          varroa   queenright
108  2022-06-02 17:13:07.060900+00:00        3693   

  0%|          | 0/380 [00:00<?, ?it/s]

,Filename,Date,Tag number,Category,Action detail,Queen status,Is alive,Report notes,target_Class
0,04-07-2022_15h45_HIVE-3693.wav,2022-07-04,3693,hive status,queenright,queenright,1,14 fob,Class 0: Active
1,04-08-2022_13h30_HIVE-3693.wav,2022-08-04,3693,hive grading,pulled honey super,queenright,1,pulled 6 honey supers,Class 0: Active
2,04-08-2022_17h30_HIVE-3628.wav,2022-08-04,3628,hive grading,pulled honey super,queenright,1,pulled 6 honey supers,Class 0: Active
3,07-06-2022_03h30_HIVE-3693.wav,2022-06-07,3693,hive status,queenless,queenless,1,saw queen cells,Class 1: Queenless
4,07-06-2022_18h45_HIVE-3690.wav,2022-06-07,3690,hive status,queenless,queenless,1,Saw multiple uncapped queens cells (10+) in th...,Class 1: Queenless
5,07-09-2022_06h15_HIVE-6.wav,2022-09-07,6,frames of bees,8,queenright,1,NaN,Class 0: Active
6,07-09-2022_22h15_HIVE-6.wav,2022-09-07,6,frames of bees,8,queenright,1,NaN,Class 0: Active
7,11-07-2022_15h45_HIVE-3631.wav,2022-07-11,3631,queen management,potential breeder,NaN,1,third box built wax already,Others / Ignored
8,11-11-2022_04h30_HIVE-3690.wav,2022-11-11,3690,NaN,NaN,queenright,1,Added V3 sensor 214,Class 0: Active
9,15-09-2022_15h00_HIVE-3631.wav,2022-09-15,3631,feeding,sugar,queenright,1,NaN,Class 0: Active


# Clean

In [13]:
# import os
# import re
# from tqdm.auto import tqdm

# frdr_audiofolder_path = r"dataset\urban-frdr\audio_2022_all_chunk"

# all_files = os.listdir(frdr_audiofolder_path)

# fix_count = 0

# for filename in tqdm(all_files):
#     if filename.endswith('.wav'):
#         old_file_path = os.path.join(frdr_audiofolder_path, filename)
#         cleaned_filename = re.sub(r'_(Active|Queenless|Infested)(_\1)+', r'_\1', filename)
        
#         if cleaned_filename != filename:
#             new_file_path = os.path.join(frdr_audiofolder_path, cleaned_filename)
            
#             if os.path.exists(old_file_path):
#                 os.rename(old_file_path, new_file_path)
#                 fix_count += 1